In [2]:
import os
import numpy as np 
import pandas as pd
from glob import glob
from datetime import datetime

In [11]:
def preprocess_data(sample, ticker, base_date):
    sample['CODE'] = ticker
    sample = sample[sample['Date'] >= base_date][['Date', 'CODE', 'Adj Close']].copy()
    sample.reset_index(inplace=True, drop=True)
    sample['STD_YM'] = sample['Date'].map(lambda x: datetime.strptime(x, '%Y-%m-%d').strftime('%Y-%m'))
    sample['1M_RET'] = 0.
    ym_keys = list(sample['STD_YM'].unique())
    return sample, ym_keys

In [37]:
def create_trade_book(sample, sample_codes):
    book = sample[sample_codes].copy()
    book['STD_YM'] = book.index.map(lambda x: datetime.strptime(x, '%Y-%m-%d').strftime('%Y-%m'))
    for c in sample_codes:
        book[f"p {c}"] = ''
        book[f"r {c}"] = ''
    return book

In [44]:
def execute_trading(book, s_codes):
    std_ym = ''; buy_phase = False;
    for s in s_codes:
        print(s)
        for idx in book.index:
            if book.loc[idx, f'p {s}'] == "" and book.shift(1).loc[idx, f"p {s}"] == f"ready {s}":
                std_ym = book.loc[idx, 'STD_YM']
                buy_phase = True
            
            if book.loc[idx, f"p {s}"] == "" and book.loc[idx, 'STD_YM'] == std_ym and buy_phase == True:
                book.loc[idx, f"p {s}"] = f"buy {s}"
            
            if book.loc[idx, f"p {s}"] == "":
                std_ym = None
                buy_phase = False

    buy_dict = dict(); sell_dict = dict();
    for idx in book.index:
        for s in s_codes:
            # long
            if book.loc[idx, f'p {s}'] == f'buy {s}' and book.shift(1).loc[idx, f'p {s}'] == f'ready {s}' and book.shift(2).loc[idx, f'p {s}'] == '':
                buy_dict[s] = book.loc[idx, s]
            elif book.loc[idx, f'p {s}'] == '' and book.shift(1).loc[idx, f'p {s}'] == f'buy {s}':
                sell_dict[s] = book.loc[idx, s]
                rtn = (sell_dict[s] / buy_dict[s]) - 1.
                book.loc[idx, f'r {s}'] = rtn
                print(f"[개별 청산일 {idx}] 종목 코드: {s}\t진입 가격: {buy_dict[s]}\t청산 가격: {sell_dict[s]}\treturn: {rtn}")
        
            if book.loc[idx, f'p {s}'] == "":
                buy_dict[s] = 0.; sell_dict[s] = 0.;
        
    acc_rtn = 1. 
    for idx in book.index:
        rtn = 0.; count = 0;
        for s in s_codes:
            if book.loc[idx, f'p {s}'] == '' and book.shift(1).loc[idx, f'p {s}'] == f'buy {s}':
                count += 1
                rtn += book.loc[idx, f'r {s}'] 
        if rtn != 0. and count != 0:
            acc_rtn *= (rtn/count) + 1.
            print(f"[누적 청산일 {idx}] 청산 종목수: {count}\t청산 수익률: {rtn}\t누적 수익률: {acc_rtn}")
        book.loc[idx, 'acc_rtn'] = acc_rtn
    print(f'[최종 누적 수익률] {acc_rtn}')
    
    return book

In [28]:
files = glob('./data/us_etf_data/*.csv')

monthly_df = pd.DataFrame(columns=['Date', 'CODE', '1M_RET'])
stock_df = pd.DataFrame(columns = ['Date', 'CODE', 'Adj Close'])

for file in files:
    if os.path.isdir(file):
        print(f"{file} <DIR>")
    else:
        folder, name = os.path.split(file)
        head, tail = os.path.splitext(name)
        read_df = pd.read_csv(file)
        price_df, ym_keys = preprocess_data(read_df, head, base_date="2010-01-02")
        stock_df = stock_df.append(price_df.loc[:, ['Date', 'CODE', 'Adj Close']], sort=False)
        
        for ym in ym_keys:
            m_ret = price_df.loc[price_df[price_df['STD_YM'] == ym].index[-1], 'Adj Close'] / price_df.loc[price_df[price_df['STD_YM'] == ym].index[0], 'Adj Close']
            price_df.loc[price_df['STD_YM'] == ym, ['1M_RET']] = m_ret
            monthly_df = monthly_df.append(price_df.loc[price_df[price_df['STD_YM'] == ym].index[-1], ['Date', 'CODE', '1M_RET']])

In [29]:
monthly_ret_df = monthly_df.pivot('Date', 'CODE', '1M_RET').copy()
monthly_ret_df = monthly_ret_df.rank(axis=1, ascending=False, method='max', pct=True)
monthly_ret_df = monthly_ret_df.where(monthly_ret_df < 0.4, np.nan)
monthly_ret_df.fillna(0, inplace=True)
monthly_ret_df[monthly_ret_df != 0] = 1.
stock_codes= list(stock_df['CODE'].unique())


In [45]:
sig_dict = dict()
for date in monthly_ret_df.index:
    ticker_list = list(monthly_ret_df.loc[date, monthly_ret_df.loc[date, :] >= 1.].index)
    sig_dict[date] = ticker_list
stock_c_matrix = stock_df.pivot('Date', 'CODE', "Adj Close").copy()
book = create_trade_book(stock_c_matrix, list(stock_df['CODE'].unique()))

for date, values in sig_dict.items():
    for stock in values:
        book.loc[date, f"p {stock}"] = f"ready {stock}"

book = execute_trading(book, stock_codes)

GDX
AMZN
GM
MSFT
USM
GLD
USO
SPY
BND
AAPL
WMT
SLV
[개별 청산일 2010-03-01] 종목 코드: GLD	진입 가격: 108.349998	청산 가격: 109.43	return: 0.009967715920031761
[개별 청산일 2010-03-01] 종목 코드: BND	진입 가격: 61.280487	청산 가격: 61.585163	return: 0.004971827328983158
[개별 청산일 2010-03-01] 종목 코드: WMT	진입 가격: 42.12096	청산 가격: 42.451756	return: 0.007853477223691119
[개별 청산일 2010-04-01] 종목 코드: GDX	진입 가격: 42.019035	청산 가격: 43.675705	return: 0.0394266550862008
[개별 청산일 2010-04-01] 종목 코드: USO	진입 가격: 38.349998	청산 가격: 41.240002	return: 0.07535864799784342
[개별 청산일 2010-04-01] 종목 코드: SPY	진입 가격: 90.145805	청산 가격: 97.770996	return: 0.08458730830569428
[개별 청산일 2010-05-03] 종목 코드: AMZN	진입 가격: 131.809998	청산 가격: 137.490005	return: 0.04309238362935108
[개별 청산일 2010-05-03] 종목 코드: USM	진입 가격: 36.212452	청산 가격: 36.744232	return: 0.014685003931796725
[개별 청산일 2010-05-03] 종목 코드: SLV	진입 가격: 17.540001	청산 가격: 18.42	return: 0.0501709777553605
[개별 청산일 2010-06-01] 종목 코드: MSFT	진입 가격: 24.56805	청산 가격: 20.70437	return: -0.15726441455467566
[개별 청산일 2010-06-01] 종목

CODE,AAPL,p AAPL,r AAPL
Date,,,
2012-01-27,55.975765,buy AAPL,
2012-01-30,56.692860,buy AAPL,
2012-01-31,57.127102,ready AAPL,
2012-02-01,57.090816,buy AAPL,
2012-02-02,56.956921,buy AAPL,
2012-02-03,57.527580,buy AAPL,
2012-02-06,58.064457,buy AAPL,
2012-02-07,58.672680,buy AAPL,
2012-02-08,59.655083,buy AAPL,
